In [ ]:
import numpy as np
import pandas as pd
import torch


In [15]:
def dataset_cleaning(data):
  df = data.copy()
  #First, we will look for the column containing the date
  date_column = None
  for col in df.columns: # We will search for the column containing dates, while requiring the user to place this column first to avoid unnecessary loops
    converted = pd.to_datetime(df[col], errors='coerce')
    if converted.notna().sum() > len(df) * 0.5:
      df[col] = converted
      date_column = col
      break

  #To ensure there are no absurd numbers that could ruin the code, we'll check if there are any and, if so, remove them
  for col in df.columns:
    if col == date_column:
      continue
    converted = pd.to_numeric(df[col], errors='coerce')
    if converted.isna().sum() > df[col].isna().sum():
      df[col] = pd.to_numeric(
          df[col].astype(str).str.replace(r'[^0-9.-]', '', regex=True),
          errors='coerce',
    )
    else:
      df[col] = converted

  sorting = df[date_column].is_monotonic_increasing # We will first check if the data is in order
  if not sorting:
    df = df.sort_values(by= date_column).reset_index(drop=True) # We will put the dates in order for better readability

  # We will use Periodic Feature Encoding so that the model can understand
  month = df[date_column].dt.month
  df['month_sin'] = np.sin(2 * np.pi * month / 12)
  df['month_cos'] = np.cos(2 * np.pi * month / 12)

  if df.isna().sum().sum() > 0 : # We apply a condition to clean the dataset if there are NaN values
    df = df.interpolate(method='linear').bfill().ffill()

  return df



In [ ]:
# Test data
date = pd.date_range(start="2026-01-01", periods=100, freq="D")
data = pd.DataFrame(
    {
        "date": date,
        "valeur_1": np.random.randn(100) * 10 + 50,
        "valeur_2": np.random.randn(100) * 5 + 20,
    }
)
df = dataset_cleaning(data)